In [30]:
import pandas as pd
import statsmodels.api as sm
from statsmodels.formula.api import ols
from statsmodels.stats.anova import anova_lm
import scipy.stats as stats
from statsmodels.stats.multicomp import pairwise_tukeyhsd




In [31]:
df = pd.read_csv("results-survey937665.csv")
df.columns.values[13] = 'group_number'

In [32]:
mapping = {
    1: ('left', 'medium', 'LLM_simple'),
    2: ('left', 'medium', 'LLM_complex'),
    3: ('right', 'medium', 'LLM_simple'),
    4: ('right', 'medium', 'LLM_complex')
}

# Apply the mapping to create new columns
df[['article_orientation', 'article_bias_level', 'explanation_type']] = df['group_number'].map(mapping).apply(pd.Series)


In [33]:
df = df.rename(columns={
    "After reading the explanation, please tell us what you think about the following sentence: In my opinion, this article is biased.": "bias_t2",
    "Please tell us what you think about the following sentence: In my opinion, this article is biased.": "bias_t1",
    "Do you consider yourself to be liberal, conservative or somewhere in between?   [Political Orientation|Liberal|Conservative]": "Participant_Political_Lean",
    "The explanation was useful. [agree|disagree]": "Usefulness",
    "The explanation was complete. [agree|disagree]": "Completeness"
})

In [34]:
relevant_columns = [
    'article_orientation', 
    'article_bias_level',             
    'explanation_type',    
    'bias_t1',
    'bias_t2',
    'Participant_Political_Lean',
    'Usefulness',
    'Completeness'
]

df_relevant = df[relevant_columns].copy()

In [35]:
bias_scale = {
    "Strongly disagree": -3,
    "Disagree": -2,
    "Somewhat disagree": -1,
    "Somewhat agree": 1,
    "Agree": 2,
    "Strongly agree": 3
}

# Map responses in bias_t1 and bias_t2 to numeric values
df_relevant['bias_t1_num'] = df_relevant['bias_t1'].map(bias_scale)
df_relevant['bias_t2_num'] = df_relevant['bias_t2'].map(bias_scale)

# Calculate the change in perceived bias
df_relevant['biaschange'] = df_relevant['bias_t2_num'] - df_relevant['bias_t1_num']

In [36]:
df_relevant.head()

,article_orientation,article_bias_level,explanation_type,bias_t1,bias_t2,Participant_Political_Lean,Usefulness,Completeness,bias_t1_num,bias_t2_num,biaschange
0,left,medium,LLM_simple,Somewhat disagree,Somewhat agree,-2.0,1.0,-2.0,-1.0,1.0,2.0
1,right,medium,LLM_simple,Somewhat disagree,NaN,0.0,NaN,NaN,-1.0,NaN,NaN
2,right,medium,LLM_complex,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,left,medium,LLM_simple,Agree,Strongly agree,-6.0,-2.0,-2.0,2.0,3.0,1.0
4,right,medium,LLM_simple,Somewhat disagree,Somewhat disagree,-6.0,-1.0,-1.0,-1.0,-1.0,0.0


First Analysis: 

H1: Generation Technique
Prediction: LLM_complex leads to greater bias perception change (BiasChange) than the LLM_simple.

Analysis:
IVs: Explanation Type (explanation_technique: LLM_complex, LLM_simple)
DV: BiasChange
Test: ANOVA
Effects Tested:
Influence of explanation_technique on BiasChange

Interaction between explanation_technique \and article_bias
Followup tests (\if significant): Tukey’s HSD \or ttests comparing:
explanation_technique: LLM_simple vs. LLM_complex

In [37]:
import pandas as pd
import statsmodels.api as sm
from statsmodels.formula.api import ols
from scipy import stats
from statsmodels.stats.multicomp import pairwise_tukeyhsd

# Make sure the categorical variables are treated as categories
df_relevant['explanation_type'] = pd.Categorical(df_relevant['explanation_type'])


# Check the unique values in your categorical variables
print(df_relevant['explanation_type'].unique())



['LLM_simple', 'LLM_complex', NaN]
Categories (2, object): ['LLM_complex', 'LLM_simple']


In [38]:
# Run ANOVA
model = ols('biaschange ~ C(explanation_type)', data=df_relevant).fit()
anova_results = anova_lm(model, typ=2)

# Print the ANOVA table
print(anova_results)

                     sum_sq    df         F    PR(>F)
C(explanation_type)   0.375   1.0  0.157068  0.700195
Residual             23.875  10.0       NaN       NaN


In [39]:
# Perform Tukey's HSD if ANOVA is significant
if anova_results['PR(>F)'][0] < 0.05:  # Check if ANOVA p-value is significant
    tukey = pairwise_tukeyhsd(endog=df_relevant['biaschange'],
                              groups=df_relevant['explanation_type'],
                              alpha=0.05)
    print(tukey.summary())
